# Modelo 3 (final) — Sistema de 9 cuerpos
## Acercamiento Tierra–Apophis 2029 con convergencia de modelo y análisis de incertidumbre

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Abril 2026

---

> **Modelo final a presentar**
>
> Este cuaderno implementa una versión consolidada del proyecto con tres bloques principales:
> 1. **Modelo dinámico de 9 cuerpos** para estimar el máximo acercamiento Tierra–Apophis.
> 2. **Convergencia de la distancia mínima** al ir añadiendo cuerpos perturbadores.
> 3. **Análisis de incertidumbre (Monte Carlo)** sobre las condiciones iniciales de Apophis.


## Alcance físico del modelo de 9 cuerpos

Para mantener un modelo completo pero computacionalmente controlado, se usa el sistema:

- Sol
- Mercurio
- Venus
- Tierra
- Luna
- Marte
- Júpiter
- Saturno
- Apophis

Esto captura el entorno gravitacional dominante del encuentro 2029 y conserva una narrativa clara para la entrega final.


In [ ]:
%pip install -Uq pymcel


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from datetime import datetime, timedelta

import pymcel as pc

print('Librerías cargadas correctamente.')


## 1) Unidades canónicas y constantes del problema

Se trabaja en unidades canónicas astronómicas con \(G=1\):

- Longitud: AU
- Masa: \(M_\odot\)
- Tiempo: \(UT = \sqrt{AU^3/(G M_\odot)}\)


In [ ]:
# ── Constantes base ───────────────────────────────────────────────────────────
AU_km    = 149_597_870.7
AU_m     = AU_km * 1e3
M_sun_kg = 1.989e30
G_SI     = 6.674e-11
R_Earth_km = 6_371.0

# GM (km^3/s^2) aproximados JPL
GM = {
    'Sol':      1.32712440018e11,
    'Mercurio': 2.2032e4,
    'Venus':    3.24858592e5,
    'Tierra':   3.986004418e5,
    'Luna':     4.9048695e3,
    'Marte':    4.282837e4,
    'Jupiter':  1.26686534e8,
    'Saturno':  3.7931187e7,
}

M_apophis_kg = 2.7e10

UT_s    = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))
UT_days = UT_s / 86400.0
vel_unit_kms = AU_km / UT_s

masas_canon = {k: v / GM['Sol'] for k, v in GM.items()}
masas_canon['Apophis'] = M_apophis_kg / M_sun_kg

print('='*64)
print(f'UT = {UT_days:.6f} días')
print(f'1 AU/UT = {vel_unit_kms:.6f} km/s')
print('Masas canónicas principales:')
for k in ['Sol','Mercurio','Venus','Tierra','Luna','Marte','Jupiter','Saturno','Apophis']:
    print(f'  {k:<9}: {masas_canon[k]:.6e}')
print('='*64)


## 2) Condiciones iniciales (NASA Horizons) en \(t_0 =\) 2028-01-01

Se consultan estados baricéntricos (`location='@0'`) y se convierten velocidades de AU/día a AU/UT.


In [ ]:
T0_STR  = '2028-01-01'
t0_date = datetime(2028, 1, 1)

# Orden final del modelo de 9 cuerpos
orden_9 = [
    'Sol', 'Mercurio', 'Venus', 'Tierra', 'Luna',
    'Marte', 'Jupiter', 'Saturno', 'Apophis'
]

ids_horizons = {
    'Sol': '10',
    'Mercurio': '199',
    'Venus': '299',
    'Tierra': '399',
    'Luna': '301',
    'Marte': '499',
    'Jupiter': '599',
    'Saturno': '699',
    'Apophis': '99942',
}

estados_raw = {}
for nombre in orden_9:
    print(f'Consultando {nombre:<9} (ID={ids_horizons[nombre]})...', end=' ')
    data = pc.consulta_horizons(
        id=ids_horizons[nombre],
        location='@0',
        epochs=T0_STR,
        datos='vectors',
        propiedades='default',
    )
    estados_raw[nombre] = data[0]
    print('OK')


def extraer_estado_canonico(df):
    row = df.iloc[0] if hasattr(df, 'iloc') else df[0]
    r = np.array([float(row['x']), float(row['y']), float(row['z'])], dtype=float)      # AU
    v = np.array([float(row['vx']), float(row['vy']), float(row['vz'])], dtype=float)    # AU/día
    return r, v * UT_days                                                                   # AU/UT


def construir_sistema(orden):
    sistema = []
    for nombre in orden:
        r, v = extraer_estado_canonico(estados_raw[nombre])
        sistema.append(dict(m=masas_canon[nombre], r=list(r), v=list(v)))
    return sistema

IDX9 = {nombre: i for i, nombre in enumerate(orden_9)}
sistema_9 = construir_sistema(orden_9)

print('\nSistema de 9 cuerpos armado correctamente.')



## 3) Integración principal y acercamiento Tierra–Apophis


In [ ]:
DURACION_DIAS = 547          # 2028-01-01 → 2029-07-01
N_PASOS = 2200

t_fin_UT = DURACION_DIAS / UT_days
ts = np.linspace(0.0, t_fin_UT, N_PASOS)
dt_dias = (ts[1] - ts[0]) * UT_days

print(f'Duración: {DURACION_DIAS} días  |  pasos: {N_PASOS}  |  dt ≈ {dt_dias:.3f} días')

rs9, vs9, rps9, vps9, const9 = pc.ncuerpos_solucion(sistema_9, ts)

r_earth = rs9[IDX9['Tierra']]
r_apoph = rs9[IDX9['Apophis']]
dist_TA_AU = np.linalg.norm(r_earth - r_apoph, axis=1)

idx_min = int(np.argmin(dist_TA_AU))
d_min_AU = float(dist_TA_AU[idx_min])
d_min_km = d_min_AU * AU_km
d_min_RE = d_min_km / R_Earth_km
fecha_min = t0_date + timedelta(days=float(ts[idx_min] * UT_days))

print('\nAcercamiento (resolución base):')
print(f'  d_min = {d_min_AU:.6e} AU = {d_min_km:,.0f} km = {d_min_RE:.2f} R_⊕')
print(f'  fecha ~ {fecha_min.strftime("%Y-%m-%d %H:%M UTC")}')



In [ ]:
# Validación simple de energía
E9 = const9['K'] + const9['U']
err_E9 = np.abs((E9 - E9[0]) / E9[0])

print(f'Error relativo max de energía: {err_E9.max():.2e}')

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogy(ts[1:] * UT_days, err_E9[1:] + 1e-20, color='steelblue')
ax.set_xlabel('Días desde 2028-01-01')
ax.set_ylabel('|ΔE/E0|')
ax.set_title('Conservación de energía — Modelo 9 cuerpos')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Refinamiento local de la fecha/distancia mínima

Se reintegra en una ventana de \(\pm 30\) días alrededor del mínimo para mejorar resolución temporal.


In [ ]:
MARGEN_DIAS = 30
N_FINO = 4500

t_ven_ini = max(0.0, ts[idx_min] - MARGEN_DIAS / UT_days)
t_ven_fin = min(t_fin_UT, ts[idx_min] + MARGEN_DIAS / UT_days)
idx_restart = int(np.argmin(np.abs(ts - t_ven_ini)))

sistema_fino = []
for nombre in orden_9:
    k = IDX9[nombre]
    sistema_fino.append(dict(
        m=masas_canon[nombre],
        r=list(rs9[k, idx_restart, :]),
        v=list(vs9[k, idx_restart, :]),
    ))

ts_rel = np.linspace(0.0, t_ven_fin - t_ven_ini, N_FINO)
rs_f, vs_f, _, _, _ = pc.ncuerpos_solucion(sistema_fino, ts_rel)

r_earth_f = rs_f[IDX9['Tierra']]
r_apoph_f = rs_f[IDX9['Apophis']]
dist_f_AU = np.linalg.norm(r_earth_f - r_apoph_f, axis=1)

idx_min_f = int(np.argmin(dist_f_AU))
d_min_f_AU = float(dist_f_AU[idx_min_f])
d_min_f_km = d_min_f_AU * AU_km
d_min_f_RE = d_min_f_km / R_Earth_km

fecha_precisa = t0_date + timedelta(days=float((t_ven_ini + ts_rel[idx_min_f]) * UT_days))

print('='*66)
print('RESULTADO FINAL (Modelo 3, 9 cuerpos)')
print('='*66)
print(f'Distancia mínima: {d_min_f_AU:.6e} AU  =  {d_min_f_km:,.0f} km  =  {d_min_f_RE:.2f} R_⊕')
print(f'Fecha estimada:   {fecha_precisa.strftime("%Y-%m-%d %H:%M UTC")}')
print('='*66)



## 4) Convergencia de la distancia mínima al añadir cuerpos

Se compara una secuencia de modelos crecientes para cuantificar la convergencia de \(d_{\min}\):

1. **SEA**: Sol–Tierra–Apophis
2. **SEMA**: + Luna
3. **SEMAJ**: + Júpiter
4. **SEMAJV**: + Venus
5. **M3-9C**: + Mercurio, Marte y Saturno (modelo final)


In [ ]:
modelos_conv = [
    ('SEA',   ['Sol', 'Tierra', 'Apophis']),
    ('SEMA',  ['Sol', 'Tierra', 'Luna', 'Apophis']),
    ('SEMAJ', ['Sol', 'Tierra', 'Luna', 'Jupiter', 'Apophis']),
    ('SEMAJV',['Sol', 'Tierra', 'Luna', 'Jupiter', 'Venus', 'Apophis']),
    ('M3-9C', orden_9),
]

N_CONV = 1400
ts_conv = np.linspace(0.0, t_fin_UT, N_CONV)

rows = []
for etiqueta, orden_m in modelos_conv:
    idx_map = {n:i for i,n in enumerate(orden_m)}
    sistema_m = construir_sistema(orden_m)
    rs_m, vs_m, _, _, _ = pc.ncuerpos_solucion(sistema_m, ts_conv)

    d_au = np.linalg.norm(rs_m[idx_map['Tierra']] - rs_m[idx_map['Apophis']], axis=1)
    i_min = int(np.argmin(d_au))
    d_km = float(d_au[i_min] * AU_km)
    fecha = t0_date + timedelta(days=float(ts_conv[i_min] * UT_days))

    rows.append(dict(
        modelo=etiqueta,
        n_cuerpos=len(orden_m),
        dmin_km=d_km,
        fecha_min=fecha,
    ))

conv_df = pd.DataFrame(rows)
conv_df['delta_vs_prev_km'] = conv_df['dmin_km'].diff()
conv_df['delta_vs_9c_km'] = conv_df['dmin_km'] - conv_df.loc[conv_df['modelo']=='M3-9C', 'dmin_km'].iloc[0]

display(conv_df)

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(conv_df['n_cuerpos'], conv_df['dmin_km'], marker='o', lw=2, color='darkblue')
for _, r in conv_df.iterrows():
    ax.annotate(r['modelo'], (r['n_cuerpos'], r['dmin_km']), textcoords='offset points', xytext=(0,8), ha='center')
ax.set_xlabel('Número de cuerpos del modelo')
ax.set_ylabel('Distancia mínima Tierra–Apophis [km]')
ax.set_title('Convergencia de d_min al añadir cuerpos')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5) Plano general de órbitas (XY): sistema completo y paso de Apophis

Se muestra una vista global del plano XY con todas las trayectorias del modelo de 9 cuerpos,
resaltando Apophis para hacer visible su paso en el intervalo 2028–2029.


In [ ]:
colores = {
    'Sol':'gold', 'Mercurio':'gray', 'Venus':'orange', 'Tierra':'deepskyblue',
    'Luna':'lightgray', 'Marte':'tomato', 'Jupiter':'sandybrown', 'Saturno':'khaki',
    'Apophis':'crimson'
}

fig, ax = plt.subplots(figsize=(9,9))
for nombre in orden_9:
    k = IDX9[nombre]
    lw = 2.8 if nombre == 'Apophis' else (1.8 if nombre in ['Tierra','Jupiter','Saturno'] else 1.1)
    alpha = 1.0 if nombre == 'Apophis' else 0.8
    ax.plot(rs9[k,:,0], rs9[k,:,1], color=colores[nombre], lw=lw, alpha=alpha, label=nombre)

# Marcas de inicio/final para Apophis
kA = IDX9['Apophis']
ax.scatter(rs9[kA,0,0], rs9[kA,0,1], color='white', s=45, edgecolors='black', zorder=10, label='Apophis t0')
ax.scatter(rs9[kA,-1,0], rs9[kA,-1,1], color='crimson', s=45, edgecolors='black', zorder=10, label='Apophis tf')

ax.set_xlabel('x [AU]')
ax.set_ylabel('y [AU]')
ax.set_title('Plano general XY — Modelo 3 (9 cuerpos)')
ax.set_aspect('equal', 'box')
ax.grid(alpha=0.25)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()


## 6) Animación 2D (HTML) en vista cercana Tierra–Apophis

Se conserva **solo una animación** en formato HTML 2D, centrada en la Tierra,
para visualizar el acercamiento local de Apophis (sin gráficas 3D).


In [ ]:
# Coordenadas geocéntricas en la ventana refinada
rE = rs_f[IDX9['Tierra']]
rA = rs_f[IDX9['Apophis']]
rM = rs_f[IDX9['Luna']]

A_geo = rA - rE
M_geo = rM - rE

N_FRAMES = 260
FPS = 25
frame_idx = np.linspace(0, len(ts_rel)-1, N_FRAMES, dtype=int)

# Límite automático en torno al encuentro
lim = max(np.percentile(np.abs(A_geo[:,0]), 99), np.percentile(np.abs(A_geo[:,1]), 99)) * 1.2
lim = max(lim, 0.0035)

fig, ax = plt.subplots(figsize=(7,7))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal')
ax.grid(alpha=0.15, color='gray')
ax.set_xlabel('x [AU]', color='white')
ax.set_ylabel('y [AU]', color='white')
ax.set_title('Vista cercana Tierra–Apophis (marco geocéntrico)', color='white')
ax.tick_params(colors='white')

# Tierra fija en origen
earth_dot, = ax.plot([0],[0], marker='o', ms=8, color='deepskyblue', markeredgecolor='white', ls='none', label='Tierra')
moon_dot,  = ax.plot([], [], marker='o', ms=4, color='lightgray', ls='none', label='Luna')
apo_dot,   = ax.plot([], [], marker='o', ms=5, color='crimson', ls='none', label='Apophis')

moon_trail, = ax.plot([], [], color='lightgray', lw=1.0, alpha=0.5)
apo_trail,  = ax.plot([], [], color='crimson',   lw=1.5, alpha=0.9)

time_txt = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='white', fontsize=10)
ax.legend(loc='upper right', facecolor='black', edgecolor='gray', labelcolor='white')

trail_len = 120

def init():
    moon_dot.set_data([], [])
    apo_dot.set_data([], [])
    moon_trail.set_data([], [])
    apo_trail.set_data([], [])
    time_txt.set_text('')
    return moon_dot, apo_dot, moon_trail, apo_trail, time_txt

def update(f):
    i = frame_idx[f]
    i0 = max(0, i - trail_len)

    moon_dot.set_data([M_geo[i,0]], [M_geo[i,1]])
    apo_dot.set_data([A_geo[i,0]], [A_geo[i,1]])

    moon_trail.set_data(M_geo[i0:i+1,0], M_geo[i0:i+1,1])
    apo_trail.set_data(A_geo[i0:i+1,0], A_geo[i0:i+1,1])

    t_abs_days = (t_ven_ini + ts_rel[i]) * UT_days
    fecha = t0_date + timedelta(days=float(t_abs_days))
    d_km = np.linalg.norm(A_geo[i]) * AU_km
    time_txt.set_text(f'{fecha:%Y-%m-%d %H:%M} UTC | d(T-A) = {d_km:,.0f} km')

    return moon_dot, apo_dot, moon_trail, apo_trail, time_txt

anim = animation.FuncAnimation(fig, update, init_func=init, frames=N_FRAMES, interval=1000/FPS, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 7) Análisis de incertidumbre: Monte Carlo en condiciones iniciales de Apophis

Se propagan perturbaciones gaussianas sobre el estado inicial de Apophis en \(t_0\):

- \(\sigma_r = 10\) km por componente en posición.
- \(\sigma_v = 10^{-4}\) km/s por componente en velocidad.

Los demás cuerpos se mantienen fijos en sus efemérides nominales.


In [ ]:
np.random.seed(42)

N_MC = 120
N_PASOS_MC = 900
ts_mc = np.linspace(0.0, t_fin_UT, N_PASOS_MC)

sigma_r_km = 10.0
sigma_v_kms = 1e-4
sigma_r_AU = sigma_r_km / AU_km
sigma_v_AUUT = sigma_v_kms / vel_unit_kms

iA = IDX9['Apophis']
rA0 = np.array(sistema_9[iA]['r'], dtype=float)
vA0 = np.array(sistema_9[iA]['v'], dtype=float)

dmins_mc_km = []
tmins_mc_days = []

for j in range(N_MC):
    dr = np.random.normal(0.0, sigma_r_AU, size=3)
    dv = np.random.normal(0.0, sigma_v_AUUT, size=3)

    sistema_j = [dict(m=b['m'], r=list(b['r']), v=list(b['v'])) for b in sistema_9]
    sistema_j[iA]['r'] = list(rA0 + dr)
    sistema_j[iA]['v'] = list(vA0 + dv)

    rs_j, _, _, _, _ = pc.ncuerpos_solucion(sistema_j, ts_mc)

    d_au = np.linalg.norm(rs_j[IDX9['Tierra']] - rs_j[IDX9['Apophis']], axis=1)
    i_min = int(np.argmin(d_au))

    dmins_mc_km.append(float(d_au[i_min] * AU_km))
    tmins_mc_days.append(float(ts_mc[i_min] * UT_days))

mc_df = pd.DataFrame({
    'dmin_km': dmins_mc_km,
    'tmin_days': tmins_mc_days,
})
mc_df['fecha_min'] = mc_df['tmin_days'].apply(lambda d: t0_date + timedelta(days=float(d)))

p16, p50, p84 = np.percentile(mc_df['dmin_km'], [16, 50, 84])
p025, p975 = np.percentile(mc_df['dmin_km'], [2.5, 97.5])

print('Resumen Monte Carlo (distancia mínima en km):')
print(f'  N = {N_MC}')
print(f'  media = {mc_df["dmin_km"].mean():,.0f} km')
print(f'  mediana = {p50:,.0f} km')
print(f'  P16-P84 = [{p16:,.0f}, {p84:,.0f}] km')
print(f'  IC 95%  = [{p025:,.0f}, {p975:,.0f}] km')

display(mc_df.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))

axes[0].hist(mc_df['dmin_km'], bins=18, color='mediumpurple', alpha=0.85, edgecolor='black')
axes[0].axvline(d_min_f_km, color='crimson', lw=2, ls='--', label='Nominal 9 cuerpos')
axes[0].set_xlabel('Distancia mínima [km]')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Monte Carlo: distribución de d_min')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].scatter(mc_df['tmin_days'], mc_df['dmin_km'], s=18, alpha=0.75, color='teal')
axes[1].set_xlabel('Día del mínimo desde 2028-01-01')
axes[1].set_ylabel('Distancia mínima [km]')
axes[1].set_title('Monte Carlo: tiempo del mínimo vs distancia')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 8) Conclusiones

- El **modelo de 9 cuerpos** entrega la estimación final de fecha y distancia mínima para abril de 2029.
- El análisis de **convergencia** permite cuantificar cuánto cambia \(d_{\min}\) al incorporar perturbadores.
- El **Monte Carlo** traduce incertidumbre inicial en una banda probabilística de acercamiento.
- La visualización se simplifica para presentación final con:
  - un **plano general XY** del sistema, y
  - **una sola animación 2D HTML** centrada en el entorno Tierra–Apophis.
